In [ ]:
import numpy as np
import pandas as pd

def entropy(y):
    values, counts = np.unique(y, return_counts=True)
    probs = counts / counts.sum()
    return -np.sum(probs * np.log2(probs))

In [ ]:
def information_gain(X, y, feature):
    parent_entropy = entropy(y)
    values, counts = np.unique(X[feature], return_counts=True)

    weighted_entropy = 0
    for v, c in zip(values, counts):
        subset_y = y[X[feature] == v]
        weighted_entropy += (c / len(y)) * entropy(subset_y)

    return parent_entropy - weighted_entropy

In [ ]:
def best_feature(X, y):
    gains = {feature: information_gain(X, y, feature) for feature in X.columns}
    return max(gains, key=gains.get)

In [ ]:
def id3(X, y):
    # All same class
    if len(np.unique(y)) == 1:
        return y.iloc[0]

    # No features left
    if X.shape[1] == 0:
        return y.value_counts().idxmax()

    best = best_feature(X, y)
    tree = {best: {}}

    for value in X[best].unique():
        sub_X = X[X[best] == value].drop(columns=best)
        sub_y = y[X[best] == value]
        tree[best][value] = id3(sub_X, sub_y)

    return tree

In [ ]:
data = {
    'Outlook': [
        'Sunny','Sunny','Overcast','Rain','Rain','Rain','Overcast','Sunny','Sunny','Rain',
        'Sunny','Overcast','Overcast','Rain','Sunny','Sunny','Overcast','Rain','Sunny','Rain',
        'Overcast','Sunny','Rain','Sunny','Overcast','Rain','Sunny','Overcast','Rain','Sunny'
    ],
    'Temperature': [
        'Hot','Hot','Hot','Mild','Cool','Cool','Mild','Cool','Mild','Mild',
        'Cool','Mild','Hot','Mild','Hot','Cool','Mild','Cool','Hot','Mild',
        'Cool','Mild','Hot','Cool','Mild','Hot','Mild','Cool','Hot','Mild'
    ],
    'Humidity': [
        'High','High','High','High','Normal','Normal','High','High','Normal','Normal',
        'High','Normal','High','Normal','High','Normal','High','Normal','High','Normal',
        'Normal','High','Normal','High','Normal','High','Normal','High','Normal','High'
    ],
    'Wind': [
        'Weak','Strong','Weak','Weak','Weak','Strong','Strong','Weak','Weak','Strong',
        'Strong','Weak','Strong','Weak','Weak','Strong','Weak','Strong','Weak','Strong',
        'Weak','Strong','Weak','Strong','Weak','Strong','Weak','Strong','Weak','Strong'
    ],
    'PlayTennis': [
        'No','No','Yes','Yes','Yes','No','Yes','No','Yes','No',
        'No','Yes','Yes','Yes','No','Yes','Yes','No','No','Yes',
        'Yes','No','Yes','No','Yes','No','Yes','Yes','No','No'
    ]
}

df = pd.DataFrame(data)

In [ ]:
X = df.drop(columns='PlayTennis')
y = df['PlayTennis']

tree = id3(X, y)
tree

{'Outlook': {'Sunny': {'Humidity': {'High': 'No', 'Normal': 'Yes'}},
  'Overcast': 'Yes',
  'Rain': {'Wind': {'Weak': {'Temperature': {'Mild': 'Yes',
      'Cool': 'Yes',
      'Hot': {'Humidity': {'Normal': 'Yes'}}}},
    'Strong': {'Temperature': {'Cool': 'No',
      'Mild': {'Humidity': {'Normal': 'No'}},
      'Hot': 'No'}}}}}}

In [ ]:
def predict(tree, sample):
    if not isinstance(tree, dict): #if leaf node. leaf node contains final class label.
        return tree

    feature = next(iter(tree))
    value = sample.get(feature)

    if value in tree[feature]:
        return predict(tree[feature][value], sample)
    else:
        return None

In [ ]:
sample = {
    'Outlook': 'Sunny',
    'Temperature': 'Cool',
    'Humidity': 'High',
    'Wind': 'Strong'
}

print("Prediction:", predict(tree, sample))

Prediction: No
